# TD 1 — Mesurer et dimensionner un service de scoring

**Le cas** : un modèle XGBoost « répond en 5 ms ». Votre client veut scorer chaque transaction bancaire
en ligne, avec ce contrat de service (SLA) :

| Exigence | Valeur |
| --- | --- |
| Réponses réussies | ≥ 99,5 % des demandes |
| Délai | p95 ≤ 100 ms côté client (au-delà de 2 s le client abandonne : échec) |
| Charge | fond 20 demandes/s, pic 100 demandes/s (30 s dans ce TD) |

**Le service** : FastAPI, livré **sans conteneur**. Une demande = une transaction. Le service la complète
avec des agrégats tirés de l'historique du client (64 features, historique de 500 000 lignes), puis
appelle le modèle.

**Votre travail** : d'abord packager le service dans une image Docker et le déployer. Ensuite, c'est
vous qui écrivez les mesures : chronométrer une demande, profiler, écrire deux générateurs de charge,
corriger le goulot, puis dimensionner.
- Les cellules marquées **À COMPLÉTER** contiennent des `...` à remplacer.
- Le module `bench` ne sert qu'à relancer le service (avec votre script de déploiement) et à fournir
  les demandes (`CORPS`).
- Les mesures durent au plus ≈ 1 min. Si une cellule tourne plus longtemps, prévenez-moi.

In [ ]:
import asyncio, cProfile, importlib, json, math, pstats, subprocess, sys, threading, time
from pathlib import Path
ROOT_DIR = Path.cwd() if (Path.cwd() / "bench.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR))   # le noyau démarre dans notebooks/
import httpx, numpy as np, pandas as pd, matplotlib.pyplot as plt
import bench, features, plots, scoring
from settings import ROOT, T0

URL = "http://localhost:8765"     # port pris ? mettre 8766
CPUS, MEMORY = 2, "4g"            # quota par défaut du conteneur
F, H = 64, 500_000                # 64 features, 6 mois d'historique
SLA = dict(succes_min=0.995, p95_max_ms=100, timeout_s=2.0)
PROFIL = [(20, 10), (100, 30), (20, 15)]   # (demandes/s, durée en s) : fond, pic, retour au fond
LIVRE = "td1-scoring:livre"       # image figée du code livré
CORPS = bench.corps(20_000)       # 20 000 demandes prêtes à envoyer (JSON, une transaction chacune)
ENTETES = {"content-type": "application/json"}

def demarrer(workers=2, cpus=CPUS, image="td1-scoring", limit=None):
    """(Re)démarre le service : `workers` processus, `cpus` CPU, `limit` demandes en cours au plus (503 au-delà)."""
    return bench.restart_server(features=F, history_rows=H, workers=workers, nthread=1,
                                cpus=cpus, memory=MEMORY, url=URL, image=image,
                                limit_concurrency=limit, label="td")

def construire(tag="td1-scoring"):
    """Construit l'image à partir de votre Dockerfile (contexte : la racine du projet)."""
    r = subprocess.run(["docker", "build", "-t", tag, str(ROOT)], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-3000:])
        raise RuntimeError("échec du build : lire l'erreur ci-dessus")

def prechauffer(n=20):
    with httpx.Client(timeout=10) as cli:
        for corps in CORPS[:n]:
            cli.post(URL + "/score", content=corps, headers=ENTETES)

def lancer(coro):
    """Exécute une coroutine asyncio depuis Jupyter, qui a déjà sa propre boucle."""
    out = {}
    def run():
        try:
            out["r"] = asyncio.run(coro)
        except BaseException as e:
            out["e"] = e
    th = threading.Thread(target=run); th.start(); th.join()
    if "e" in out:
        raise out["e"]
    return out["r"]

pd.set_option("display.width", 200, "display.max_columns", 30)
T_START = time.perf_counter()
print("machine :", bench.machine_info())

## Étape préliminaire — Packager et déployer le service (≈ 25 min)

Le dépôt contient le service (`app.py`, `features.py`, `scoring.py`, `settings.py`), les modèles
(`models/`) et les dépendances figées (`pyproject.toml`, `uv.lock`). Il n'y a ni image ni script de
lancement : vous les écrivez. Pour lancer le service à la main, hors conteneur :
`uv run uvicorn app:app --port 8000`.

**a. L'image.** Écrivez le `Dockerfile` et le `.dockerignore` (cellules `%%writefile` ci-dessous).
Contrat de l'image :
- base `python:3.13-slim`, dépendances installées avec uv depuis `uv.lock`, sans les dépendances de
  développement (guide : https://docs.astral.sh/uv/guides/integration/docker/) ;
- XGBoost a besoin d'OpenMP : paquet Debian `libgomp1` ;
- l'image contient le code (`*.py`) et `models/`, **pas** les données : elles sont montées au lancement
  dans `/app/data`, et le service les cherche dans le dossier indiqué par la variable `DATA_DIR` ;
- commande : `uvicorn app:app` sur `0.0.0.0:8000`. Configuration par variables d'environnement :
  `FEATURES`, `HISTORY_ROWS`, `XGB_NTHREAD` (lues par `settings.py`), `WEB_CONCURRENCY` (nombre de
  workers, lu par uvicorn) et `UVICORN_LIMIT_CONCURRENCY` (option `--limit-concurrency`, lue par uvicorn) ;
- le `.dockerignore` écarte ce qui n'a rien à faire dans le contexte de build.

In [ ]:
%%writefile {ROOT}/Dockerfile
# À COMPLÉTER : image du service de scoring


In [ ]:
%%writefile {ROOT}/.dockerignore
# À COMPLÉTER


In [ ]:
t = time.perf_counter()
construire()
subprocess.run(["docker", "tag", "td1-scoring", LIVRE], check=True)   # copie figée du code livré
print(f"image construite en {time.perf_counter() - t:.0f} s")
print(subprocess.run(["docker", "image", "ls", "td1-scoring"], capture_output=True, text=True).stdout)

**Question.** Quelle est la taille de l'image ? Qu'est-ce qui la gonfle (`docker history td1-scoring`) ?
Le build suivant, après une modification de `features.py`, réinstalle-t-il les dépendances ? Pourquoi ?

**b. Le déploiement.** Écrivez `scripts/deploy.sh`. `demarrer(...)`, utilisé dans tout le TD,
l'appelle avec ces variables d'environnement :

| Variable | Sens |
| --- | --- |
| `IMAGE` | image à lancer (`td1-scoring` ou `td1-scoring:livre`) |
| `PORT` | port de la machine hôte où le service doit répondre |
| `ROOT` | racine du projet ; les données sont dans `$ROOT/data` |
| `WEB_CONCURRENCY`, `XGB_NTHREAD`, `FEATURES`, `HISTORY_ROWS` | configuration du service, à transmettre au conteneur |
| `CPUS`, `MEMORY` | quota du conteneur (ce que vous payez) |
| `LIMIT` | vide : pas de limite ; sinon nombre maximal de demandes en cours |

- `bash scripts/deploy.sh` remplace le conteneur nommé `td1-api` par un nouveau, en arrière-plan, et
  rend la main tout de suite (`demarrer` attend ensuite que `/health` réponde) ;
- `bash scripts/deploy.sh stop` arrête `td1-api`.

In [ ]:
%%writefile {ROOT}/scripts/deploy.sh
#!/usr/bin/env bash
# À COMPLÉTER
set -euo pipefail


In [ ]:
demarrer(image=LIVRE)
print("quota (cpus, mémoire) :", bench.container_limits(), "| workers :", bench.container_env("WEB_CONCURRENCY"))
with httpx.Client(timeout=10) as cli:
    print("health :", cli.get(URL + "/health").json())
    r = cli.post(URL + "/score", content=CORPS[0], headers=ENTETES)
    print("score  :", r.status_code, r.json()["scores"])

**Vérifier.** Le quota vaut-il `('2', '4g')` et le nombre de workers 2 ? `/health` annonce-t-il
`features` 64 et `history_rows` 500 000 ? Si `demarrer` échoue : `docker logs td1-api`.

## Étape 0 — Chronométrer une demande (≈ 8 min)

L'API : `POST URL + "/score"`, corps `CORPS[i]`, en-têtes `ENTETES`. La réponse contient :
- l'en-tête `X-Server-Total-Ms` : temps passé dans le serveur ;
- le JSON `{"scores": [...], "timings_ms": {"enrich": …, "matrix": …, "predict": …, "handler": …}}` :
  `enrich` calcule les agrégats historiques, `matrix` met en forme, `predict` appelle XGBoost.

Écrivez `mesurer`, puis envoyez 50 demandes au service livré (2 workers, 2 CPU).

In [ ]:
# À COMPLÉTER
def mesurer(cli, corps):
    """Envoie UNE demande avec le client httpx `cli` et renvoie un dict (durées en ms) :
    latence_ms (chronométrée par vous avec time.perf_counter, côté client), serveur_ms (en-tête
    X-Server-Total-Ms), enrich_ms, matrix_ms, predict_ms (champ timings_ms de la réponse JSON)."""
    ...

demarrer(image=LIVRE)
prechauffer()
with httpx.Client(timeout=10) as cli:
    m0 = pd.DataFrame([mesurer(cli, CORPS[i]) for i in range(50)])
m0.quantile([0.5, 0.95]).round(2).rename(index={0.5: "p50", 0.95: "p95"})

**Conclure (2 phrases).** Quelle part de la latence le modèle représente-t-il ? Si on le rendait
10 fois plus rapide, combien de pourcents de latence gagnerait-on ? Où faut-il chercher le gain ?

## Étape 1 — Trouver le goulot (≈ 10 min)

**1.a Profiler.** Le scoring s'appelle aussi directement, sans HTTP :
`scoring.score(tx, hist, modele, schema, T0)`. Profilez-le avec `cProfile`, puis affichez les fonctions
les plus coûteuses avec `pstats` (triées par temps cumulé).

In [ ]:
hist, modele, schema = bench.load_direct(H, F, 1)   # historique, modèle, schéma de features
tx = bench.reserve(30)                              # 30 transactions
# À COMPLÉTER : profilez scoring.score(tx, hist, modele, schema, T0), affichez les 15 premières lignes
...

**1.b Prédire, puis mesurer.** Lisez `query` dans `features.py` : combien de lignes d'historique
examine-t-elle à chaque appel ? Prédisez le rapport des temps d'`enrich` entre H = 500 000 et H = 50 000,
puis écrivez la mesure.

In [ ]:
HS = (50_000, 150_000, 500_000)
PREDICTION_H = ...   # À COMPLÉTER : temps(H=500 000) / temps(H=50 000) attendu

# À COMPLÉTER
def cout_enrich(h, n=100):
    """ms d'enrich par transaction, en appel direct sur n transactions avec un historique de h lignes.
    bench.load_direct(h, F, 1) donne (hist, modele, schema) ; scoring.score renvoie (ids, scores, durees),
    durees.enrich est en secondes pour les n transactions."""
    ...

cout = pd.DataFrame({"h": HS, "ms_par_tx": [cout_enrich(h) for h in HS]})
print("rapport mesuré :", round(cout.ms_par_tx.iloc[-1] / cout.ms_par_tx.iloc[0], 1), "| prédit :", PREDICTION_H)
cout

**1.c Et la qualité ?** Réduire H rendrait chaque demande moins chère, mais les agrégats ne
décriraient plus qu'une période plus courte. Ce tableau a été mesuré à l'avance sur 1 000 000 de
transactions jamais vues à l'entraînement (2 % de fraudes) ; le modèle a été entraîné avec H = 500 000.
AUC : 0,5 = hasard, 1 = parfait ; un écart de moins de 0,005 est dans le bruit.

In [ ]:
q = json.loads((ROOT / "models" / "QUALITY.json").read_text())
qualite = pd.DataFrame(q["rows"])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for f, g in qualite.groupby("f"):
    ax[0].plot(g.h, g.auc, "o-", label=f"F={f}")
ax[0].axhline(q["sans_historique"]["auc"], color="k", ls=":", label="sans historique")
ax[0].set_xscale("log"); ax[0].set_xlabel("H (lignes d'historique)"); ax[0].set_title("qualité : AUC"); ax[0].legend()
if isinstance(cout, pd.DataFrame) and "ms_par_tx" in cout and cout.ms_par_tx.notna().all():
    ax[1].plot(cout.h, cout.ms_par_tx, "o-", color="C3")
ax[1].set_xscale("log"); ax[1].set_xlabel("H (lignes d'historique)"); ax[1].set_title(f"coût : enrich par transaction (ms), F={F}")
plt.tight_layout(); plt.show()
qualite.pivot(index=["h", "jours"], columns="f", values="auc")

**Conclure.** Le coût suit-il votre prédiction ? Pourquoi pas exactement (indice : ce qui ne dépend
pas de H) ? Pour tenir le SLA, diviser H par 5 serait-il une bonne idée ? Et F = 256 vaut-il son coût ?

## Étape 2 — Supprimer le travail répété (≈ 10 min)

Le service relit tout l'historique pour retrouver les quelques lignes d'**un** client. Dans `features.py`,
complétez `client_index` et `client_rows` (≈ 5 lignes ; les docstrings donnent les indices), enregistrez,
puis exécutez la cellule suivante. `enrich` utilise l'index dès que `client_index` ne renvoie plus `None`.

In [ ]:
importlib.reload(features)
print("scores inchangés (500 transactions, 3 largeurs) :", bench.check_golden())
construire()
print("image td1-scoring reconstruite avec votre code")

In [ ]:
# votre mesure de l'étape 1, rejouée sur le code corrigé
cout_corrige = pd.DataFrame({"h": HS, "ms_par_tx": [cout_enrich(h) for h in HS]})
cout.assign(corrige_ms_par_tx=cout_corrige.ms_par_tx, gain=(cout.ms_par_tx / cout_corrige.ms_par_tx).round(1))

**Conclure.** Le temps dépend-il encore de H ? Que coûte l'index (calcul une fois, mémoire) et
quand faut-il le reconstruire ?

## Étape 3 — Capacité, file d'attente et pic (≈ 30 min)

**3.a K clients qui attendent leur réponse.** Chaque client envoie une demande, attend la réponse, puis
recommence. Écrivez ce test, puis lancez-le sur le service livré avec 1, 2, 4, 8 et 16 clients.

In [ ]:
# À COMPLÉTER
def client(fin, latences):
    """Un client : tant que time.perf_counter() < fin, envoie une demande (CORPS[i % len(CORPS)]),
    attend la réponse et ajoute sa latence en ms à la liste `latences`."""
    with httpx.Client(timeout=SLA["timeout_s"]) as cli:
        i = 0
        while time.perf_counter() < fin:
            ...
            i += 1

def test_clients(k, duree_s=6):
    """k clients en parallèle (un threading.Thread chacun) pendant duree_s secondes."""
    fin = time.perf_counter() + duree_s
    latences = []
    ...   # créer les k threads (target=client, args=(fin, latences)), les démarrer, attendre leur fin
    if not latences:
        return {"k": k}
    return {"k": k, "debit": len(latences) / duree_s, "latence_moy_ms": np.mean(latences),
            "p95_ms": np.percentile(latences, 95)}

demarrer(image=LIVRE)
prechauffer()
capa = pd.DataFrame([test_clients(k) for k in (1, 2, 4, 8, 16)])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(capa.k, capa.debit, "o-"); ax[0].set_title("débit (req/s)")
ax[1].plot(capa.k, capa.latence_moy_ms, "o-"); ax[1].set_title("latence moyenne (ms)")
for a in ax: a.set_xlabel("clients simultanés k")
plt.tight_layout(); plt.show()
capa.round(1)

**Conclure.** Quelle est la capacité du service livré ? Comparez-la à la théorie : 2 workers qui
mettent chacun la latence de k = 1 par demande. Pourquoi la latence grimpe-t-elle une fois le débit
plafonné ?

**3.b Les transactions n'attendent pas.** En 3.a, la charge s'adaptait au service : un client lent
envoie moins. En production, les transactions arrivent à **leur** rythme. Écrivez un générateur qui
envoie chaque demande à une heure fixée d'avance, que le service ait répondu ou non, et qui compte la
latence **depuis l'heure prévue**. Le squelette asyncio est fourni ; complétez `plan_envois` et `envoyer`.

*Bloqué depuis 15 min ? Mettez `SECOURS = True` dans la cellule suivante : les mesures utiliseront alors
`bench.charge`, le générateur de secours.*

In [ ]:
# À COMPLÉTER
def plan_envois(profil):
    """Liste de tuples (palier, debit, t_prevu) : pour chaque palier (debit, duree) de profil, une demande
    toutes les 1/debit secondes ; les paliers s'enchaînent (t_prevu en s depuis le début du test)."""
    ...

async def envoyer(cli, t0, palier, debit, t_prevu, corps, resultats):
    """Attend l'heure t0 + t_prevu (asyncio.sleep), envoie `corps` avec `await cli.post(...)`, puis ajoute à
    `resultats` un dict : palier, debit_demande, t_prevu, t_fin (s depuis t0), latence_ms (depuis l'heure
    PRÉVUE), statut ("ok", "timeout" si httpx.TimeoutException, sinon f"http{code}")."""
    ...

async def _charge(profil):   # fourni : lance tous les envois en parallèle
    resultats = []
    limites = httpx.Limits(max_connections=None, max_keepalive_connections=200)
    async with httpx.AsyncClient(timeout=SLA["timeout_s"], limits=limites) as cli:
        t0 = time.perf_counter() + 0.2
        await asyncio.gather(*(envoyer(cli, t0, k, r, tp, CORPS[i % len(CORPS)], resultats)
                               for i, (k, r, tp) in enumerate(plan_envois(profil))))
    df = pd.DataFrame(resultats)
    return df.sort_values("t_prevu", ignore_index=True) if len(df) else df

def ma_charge(profil):
    prechauffer()
    return lancer(_charge(profil))

essai_20 = ma_charge([(20, 3)])   # 60 demandes à 20/s : le service livré doit toutes les servir
print(essai_20.statut.value_counts().to_dict(), "| latence p50 :", round(essai_20.latence_ms.median(), 1), "ms")

In [ ]:
SECOURS = False   # True si votre générateur ne fonctionne pas : on utilise bench.charge
def charger(profil):
    """Votre générateur, ou le générateur de secours si SECOURS = True."""
    if SECOURS:
        return bench.charge(profil, timeout_s=SLA["timeout_s"], url=URL, label="secours")
    return ma_charge(profil)

**3.c Le pic.** D'après la capacité mesurée en 3.a, le service livré tiendra-t-il `PROFIL`
(fond 20/s, pic 100/s pendant 30 s, retour au fond) ? Notez votre prédiction, puis lancez.

In [ ]:
PREDICTION_PIC = ...   # À COMPLÉTER : True ou False
demarrer(image=LIVRE)
avant = charger(PROFIL)

Chaque ligne de `avant` est une demande : `t_prevu`, `t_fin`, `latence_ms`, `statut`. Écrivez
deux fonctions d'analyse : la taille de la file à l'instant `t`, et le verdict du SLA.

In [ ]:
# À COMPLÉTER
def en_attente(df, t):
    """Nombre de demandes envoyées avant t et pas encore terminées à t."""
    ...

def respecte_sla(df, sla):
    """{"succes": part des demandes réussies, "p95_ms": p95 des réussies, "respecte": bool}"""
    ...

print("service livré :", respecte_sla(avant, SLA))
plots.timeline(avant, SLA, en_attente, title="code livré, 2 CPU"); plt.show()

**Lire la courbe.** Quand les arrivées dépassent la capacité, de combien de demandes par seconde
la file grossit-elle ? Que se passe-t-il **après** le pic, quand la charge revient à 20/s ? Pourquoi
le service continue-t-il à échouer alors qu'il a la capacité de suivre le fond ?

**3.d Défi : tenir `PROFIL` au moindre coût.** Leviers : `image` (votre code corrigé `"td1-scoring"`
ou le livré `LIVRE`), `workers`, `cpus` (ce que vous payez) et `limit` (au-delà de `limit` demandes en
cours, le service répond 503 tout de suite au lieu de les mettre en file). Un essai dure ≈ 1 min ;
trois essais suffisent. Essayez au moins une fois `limit` sur le service livré.

In [ ]:
essais = []
def essai(image="td1-scoring", workers=1, cpus=1, limit=None):
    demarrer(workers=workers, cpus=cpus, image=image, limit=limit)
    df = charger(PROFIL)
    plots.timeline(df, SLA, en_attente, title=f"{image}, {workers} worker(s), {cpus} CPU, limit={limit}"); plt.show()
    essais.append(dict(image=image, workers=workers, cpus=cpus, limit=limit, **(respecte_sla(df, SLA) or {})))
    return pd.DataFrame(essais)

In [ ]:
# À COMPLÉTER : vos essais, un par ligne, par exemple essai(workers=1, cpus=1)
...

**Conclure.** Quelle configuration respecte le SLA au moindre coût ? Avec `limit`, que devient
l'après-pic, et pourquoi ? Rejeter vite vaut-il mieux qu'attendre longtemps ?

## Étape 4 — Dimensionner (≈ 7 min)

Le client annonce un pic 3 fois plus fort l'an prochain. Trouvez la capacité utile d'**un** réplica
(code corrigé, 1 worker, 1 CPU) : montez la charge par paliers jusqu'à ce qu'il décroche. Choisissez vos
paliers (6 s chacun suffisent), puis écrivez `dimensionner`.

In [ ]:
PALIERS = ...   # À COMPLÉTER : [(débit en req/s, durée en s), ...] croissants

In [ ]:
demarrer(workers=1, cpus=1)
paliers = charger(PALIERS)
table = bench.par_palier(paliers).join(
    pd.DataFrame([respecte_sla(g, SLA) or {} for _, g in paliers.groupby("palier")]))
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
ax[0].plot(table.debit_demande, table.debit_realise, "o-"); ax[0].plot(table.debit_demande, table.debit_demande, "k:", lw=1); ax[0].set_title("débit réalisé")
if "p95_ms" in table:
    ax[1].plot(table.debit_demande, table.p95_ms, "o-"); ax[1].axhline(SLA["p95_max_ms"], color="k", ls="--"); ax[1].set_yscale("log")
    ax[2].plot(table.debit_demande, 100 * (1 - table.succes), "o-")
ax[1].set_title("p95 (ms)"); ax[2].set_title("échecs (%)")
for a in ax: a.set_xlabel("débit demandé (req/s)")
plt.tight_layout(); plt.show()
table

In [ ]:
MARGE, PIC_FUTUR = 0.30, 3 * 100
# À COMPLÉTER
def dimensionner(table, pic, marge):
    """Renvoie (capacité utile d'un réplica en req/s, nombre de réplicas pour absorber `pic`).
    Capacité utile = plus haut débit demandé dont le palier respecte le SLA (colonne `respecte`),
    diminuée de la marge."""
    ...

cap, n = dimensionner(table, PIC_FUTUR, MARGE)
print(f"capacité utile d'un réplica 1 CPU : {cap:.0f} req/s -> {n} réplicas pour {PIC_FUTUR} req/s")

### Restitution (5 lignes)
1. Ce qui limitait le service livré, et la mesure qui le prouve.
2. La correction et son effet mesuré (avant/après), scores inchangés.
3. Le comportement au pic : file, latence, échecs, retour à la normale ; ce qu'apporte `limit`.
4. Le dimensionnement proposé pour le pic ×3 (CPU, réplicas, marge) et ce qu'il coûterait.
5. Ce que vous ne savez pas encore (réseau réel, données réelles…) et comment le mesurer.

In [ ]:
print(f"Durée du notebook : {(time.perf_counter() - T_START) / 60:.1f} min")
bench.stop_server()